In [ ]:
#RAG

## 1. Loader

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv(override=True)
print("OPENAI_API_KEY configured:", bool(os.getenv("OPENAI_API_KEY")))

In [ ]:
from langchain_community.document_loaders import PyMuPDFLoader

file_path = "docs/260109_26년 경제성장전략_최종.pdf"
loader = PyMuPDFLoader(file_path)
docs = loader.load()

len(docs)

In [ ]:
dict(docs[37])

## 2. Splitter

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 100
)

chunkks = text_splitter.split_documents(docs)
len(chunkks)

In [ ]:
dict(chunkks[0])

## 3. VectorStore
- embedding
- vectorStore

In [ ]:
print("UPSTAGE_API_KEY configured:", bool(os.getenv("UPSTAGE_API_KEY")))

In [ ]:
from langchain_upstage import UpstageEmbeddings

embeddings = UpstageEmbeddings(
    model = "solar-embedding-1-large"
)

In [ ]:
result = embeddings.embed_documents(
    [
        "대한민국의 수도는 서울입니다.",
        "대한민국의 남쪽엔 부산이 있습니다."
    ]
)

In [ ]:
len(result[0])

### FAISS

In [ ]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-large",
    # With the `text-embedding-3` class
    # of models, you can specify the size
    # of the embeddings you want returned.
    # dimensions=1024
)

In [ ]:
'''
from langchain_community.vectorstores import FAISS

FAISS.from_documents(
    documents = chunkks,
    embedding = embeddings
)
'''

In [ ]:
vectorstore = FAISS.from_documents(
    documents = chunkks, # 처음 3개만 테스트
    embedding = embeddings
)

- vetorstore test

In [ ]:
query = "2026년 대한민국의 실질 GDP 성장률에 대한 정부의 구체적인 전망치는 무엇입니까?"
results = vectorstore.similarity_search(query, k = 3)  
for doc in results:
    print(doc.page_content[:50])

In [ ]:
results = vectorstore.similarity_search_with_score(query, k = 3)
doc, score = results[0]
print(f"점수: {score}")

##RAG 구성

In [ ]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template("""
You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.
Question: {question} 
Context: {context} 
Answer: 
""")

In [ ]:
from langchain.chat_models import init_chat_model
from langchain_core.output_parsers import StrOutputParser

llm = init_chat_model(
    model = "gpt-5-nano",
    temperature = 0,

)

chain = prompt | llm | StrOutputParser()

In [ ]:
query = "2026년 대한민국의 실질 GDP 성장률에 대한 정부의 구체적인 전망치는 무엇입니까?"

retriver = vectorstore.as_retriever(search_kwargs={'k':3})
retrieved_docs = retriver.invoke(query)
docs_str = "\n\n".join(doc.page_content for doc in retrieved_docs)

user_inputs = {
    "question": query, "context": docs_str
}

response = chain.invoke(user_inputs)
response

### chain 데코레이터

In [ ]:
from langchain_core.runnables import chain

# retriever 정의 추가
retriever = vectorstore.as_retriever()

# 그 다음 qa 실행
result = qa.invoke(query)

@chain
def qa(query):
    retrieved_docs = retriever.invoke(query)
   
    combined_docs = "\n\n".join(doc.page_content for doc in retrieved_docs)
    
    formatted = prompt.invoke({'context': combined_docs, 'question': query})
    
    answer = llm.invoke(formatted)
    return answer

query = "2026년 대한민국의 실질 GDP 성장률에 대한 정부의 구체적인 전망치는 무엇입니까?"
result = qa.invoke(query)
result

### retriever를 chain에 포함

In [ ]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# 1. 문서들을 하나의 문자열로 합치는 보조 함수
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# 2. 체인 구성
# 이 체인은 (질문) -> (컨텍스트 검색 및 포맷팅) -> (프롬프트 결합) -> (LLM 실행) -> (결과 파싱) 순으로 작동합니다.
rag_chain = (
    {
        "context": retriever | format_docs,  # 질문을 받아 검색하고 문자열로 합침
        "question": RunnablePassthrough()    # 사용자의 질문을 그대로 전달
    }
    | prompt                                 # 'context'와 'question'을 프롬프트에 주입
    | llm                                    # 모델 실행
    | StrOutputParser()                      # 결과에서 텍스트만 추출
)

# 3. 체인 실행
query = "2026년 대한민국의 실질 GDP 성장률에 대한 정부의 구체적인 전망치는 무엇입니까?"
result = rag_chain.invoke(query)
result